Before you turn this problem in, make sure everything runs as expected. First, **restart the kernel** (in the menubar, select Kernel$\rightarrow$Restart) and then **run all cells** (in the menubar, select Cell$\rightarrow$Run All).

Make sure you fill in any place that says `YOUR CODE HERE` or "YOUR ANSWER HERE", as well as your name and collaborators below:

NAME = "Nicole Paraschiv"
COLLABORATORS = ""

---

# CSCI 3155 Spring 2026: Assignment 9

Goals of this assignment include: 
- Developing a type system for records and references

In [1]:
// TEST HELPER
def passed(points: Int) = {
    require(points >=0)
    if (points == 1) print(s"\n*** Tests Passed (1 point) ***\n")
    else print(s"\n*** Tests Passed ($points points) ***\n")
}

defined function passed

## Problem 1 (50 points)

In this problem, we will extend Lettuce language and its type system to records and mutable references.  Here is a stripped down version of Lettuce we are going to use in this problem.

$$\begin{array}{rcl}
\mathbf{Expr} & \rightarrow & Const(\mathbf{Double}) \\
& | & Ident(\mathbf{Identifier})\\
& | & Plus(\mathbf{Expr}, \mathbf{Expr})\\
& | & Div(\mathbf{Expr}, \mathbf{Expr}) \\
& | & Geq(\mathbf{Expr}, \mathbf{Expr}) \\
& | & And(\mathbf{Expr}, \mathbf{Expr}) \\
& | & IfThenElse(\mathbf{Expr}, \mathbf{Expr}, \mathbf{Expr})\\
& | & Let(\mathbf{Identifier}, \color{red}{\mathbf{Type}}, \mathbf{Expr}, \mathbf{Expr})\\
& | & FunDef(\mathbf{Identifier}, \color{red}{\mathbf{Type}}, \mathbf{Expr}) \\
& | & FunCall(\mathbf{Expr}, \mathbf{Expr}) \\
& | & \color{blue}{UnitConst} & \leftarrow \text{written as} \ () \\
& | & \color{blue}{NewRef(\mathbf{Expr})} \\
& | & \color{blue}{DeRef(\mathbf{Expr})} \\
& | & \color{blue}{AssignRef(\mathbf{Expr}, \mathbf{Expr})} & \leftarrow \text{written as} \ e_1 := e_2 \\
& | & \color{blue}{Record(List[(Identifier, \mathbf{Expr})])} & \leftarrow \text{written as} \ \{\overline{x=e}\}, \ \text{e.g.,} \ \{x_1=e_1, x_2=e_2, \ldots, x_i=e_i, \ldots \} \\
\end{array}$$

We extend the type system with a record type constructor. 
$$\begin{array}{rcl}
\mathbf{Type} & \rightarrow & NumType & \leftarrow \text{written as } \ num \\
& | & BoolType & \leftarrow \text{written as } \ bool \\
& | & UnitType & \leftarrow \text{written as } \ unit \\
& | & FunType(\mathbf{Type} , \mathbf{Type} ) & \leftarrow \text{written as } \ t_1 \Rightarrow t_2 \\
& | & \color{red}{RefType(\mathbf{Type})} & \leftarrow \text{written as } \ ref(t_1) \\
& | & \color{red}{RecordType(List[(Identifier, \mathbf{Type})])} & \leftarrow \text{written as} \ \{\overline{x:t}\}, \ \text{e.g.,} \ \{x_1:t_1, x_2:t_2, \ldots, x_i:t_i, \ldots \} \\  
\end{array}$$




### A (10 points; Manually Graded): Let us write some inference rules for working with records and ref types.

$$\newcommand\typeOf{\mathbf{typeOf}}$$
$$\newcommand\semRule[3]{\begin{array}{c} #1 \\ \hline #2 \\ \end{array}\;(\text{#3}) }$$
Recall from notes on "Types and Type Checking" that $\typeOf(\texttt{e}, \alpha)$ is the type of an 
expression $\texttt{e}$ under type environment $\alpha$. The type environment maps identifiers in the current scope to  their annotated types.

Let us write the type rule for the record expression. The rule is straightforward; it simply composes the types of all sub-expressions that make up the record.

$$\semRule{ \forall i.~\typeOf(e_i, \alpha) = t_i,\;\; t_i \not= \mathbf{typeerror} }
          {\typeOf(\{\overline{x=e}\}, \alpha) = \{\overline{x:t}\} }{record-ok}$$



Next, let us write a rule for NewRef.

$$\semRule{ \typeOf(\texttt{e}, \alpha) = t,\; t \not= \mathbf{typeerror} }{\typeOf(\texttt{NewRef(e)}, \alpha) = ref(t) }{newref-ok}$$

It says that if $\texttt{e}$ receives type $t$ under type environment $\alpha$ and it is not a type error, then $\texttt{NewRef(e)}$ must receive the type $ref(t)$ under $\alpha$.



(i) Complete the missing terms for the rule for `UnitConst` OK rule. The unit constant, i.e., `()`, always has the type `unit`.
$$\semRule{}{\typeOf((), \alpha) = \color{red}{???_1}}{unitconst-ok}$$

(ii) Complete the missing terms for the rule for `DeRef` OK rule. For example, dereferencing a reference to a number (`num`) should give you a number (`num`).
$$\semRule{\typeOf(\texttt{e}, \alpha) = ref(t)}{\typeOf(\texttt{DeRef(e)}, \alpha) = \color{red}{???_2}}{deref-ok}$$

(iii) Complete the missing terms for the rule for `DeRef` error.
$$\semRule{\typeOf(\texttt{e}, \alpha) \not= \color{red}{???_3} }{\typeOf(\texttt{DeRef(e)}, \alpha) = \mathbf{typeerror}}{deref-nok}$$

(iv) Complete the missing terms for `AssignRef` OK rule.
$$\semRule{\typeOf(\texttt{e1}, \alpha) = \color{red}{???_4}, \typeOf(\texttt{e2}, \alpha) = t}{\typeOf(\texttt{AssignRef(e1, e2)}, \alpha) =   unit}{assignref-ok}$$


In the cell below, write down the solution for each of the missing terms $\color{red}{???_1}$ - $\color{red}{???_4}$. Write each one on its own line.

1. unit
2. t
3. ref(t)
4. ref(t)

### C (15 points): Write the type checker with references by filling in the missing code.

In [5]:
sealed trait Type
case object NumType extends Type
case object BoolType extends Type
case class FunType(t1: Type, t2: Type) extends Type
// TODO: Write new case classes for UnitType, RefType, and RecordType
// YOUR CODE HERE
case object UnitType extends Type
case class RefType(t: Type) extends Type
case class RecordType(ies: List[(String,Type)]) extends Type

sealed trait Program
sealed trait Expr

case class Const(f: Double) extends Expr
case class Ident(s: String) extends Expr
case class Plus(e1: Expr, e2: Expr) extends Expr
case class Minus(e1: Expr, e2: Expr) extends Expr
case class Geq(e1: Expr, e2: Expr) extends Expr
case class IfThenElse(e1: Expr, e2: Expr, e3: Expr) extends Expr
case class Let(x: String, xType: Type, e1: Expr, e2: Expr) extends Expr
case class FunDef(id: String, idType: Type, e: Expr) extends Expr
case class FunCall(calledFun: Expr, argExpr: Expr) extends Expr
case class NewRef(e: Expr) extends Expr
case class DeRef(e: Expr) extends Expr
case class AssignRef(e1: Expr, e2: Expr) extends Expr
case object UnitConst extends Expr
case class Record(ies: List[(String,Expr)]) extends Expr
case class TopLevel(e: Expr) extends Program

def typeEquals(t1: Type, t2: Type): Boolean = t1 == t2
case class TypeErrorException(s: String) extends Exception


defined trait Type
defined object NumType
defined object BoolType
defined class FunType
defined object UnitType
defined class RefType
defined class RecordType
defined trait Program
defined trait Expr
defined class Const
defined class Ident
defined class Plus
defined class Minus
defined class Geq
defined class IfThenElse
defined class Let
defined class FunDef
defined class FunCall
defined class NewRef
defined class DeRef
defined class AssignRef
defined object UnitConst
defined class Record
defined class TopLevel
defined function typeEquals
defined class TypeErrorException

In [6]:
def typeOf(e: Expr, alpha: Map[String, Type]): Type = {
    def checkType(opName: String, e1: Expr, t1: Type, e2: Expr, t2: Type, resType: Type): Type = {
        val t1hat = typeOf(e1, alpha)
        if (! typeEquals(t1hat, t1)){
            throw new TypeErrorException(s"Type mismatch in arithmetic/comparison/bool op $opName, Expected type $t1, obtained $t1hat")
        }
        
        val t2hat = typeOf(e2, alpha)
        if (! typeEquals(t2hat, t2)){
            throw new TypeErrorException(s"Type mismatch in arithmetic/comparison/bool op $opName, Expected type $t2, obtained $t2hat")
        }
        
        resType
    }
    
    e match {
        case Const(f) => NumType
        case Ident(s) => {if (alpha contains s)
                             alpha(s)
                          else 
                             throw TypeErrorException(s"Unknown identifier $s")}
        case Plus(e1, e2) =>  checkType("Plus", e1,  NumType, e2, NumType, NumType)
        case Minus(e1, e2) => checkType("Minus",e1,  NumType, e2, NumType, NumType)
        case Geq(e1, e2) => checkType("Geq", e1,  NumType, e2, NumType, BoolType)
        case IfThenElse(e, e1, e2) => {
            val t = typeOf(e, alpha)
            if (t == BoolType){
                val t1 = typeOf(e1, alpha)
                val t2 = typeOf(e2, alpha)
                if (typeEquals(t1, t2))
                    t1
                else 
                    throw TypeErrorException(s"If then else returns unequal types $t1 and $t2")
            } else {
                throw TypeErrorException(s"If then else condition expression not boolean $t")
            }
        }

        case Let(x, t, e1, e2) => {
            val t1 = typeOf(e1, alpha)
            if (typeEquals(t1, t)){
                val newAlpha = alpha + (x -> t)
                typeOf(e2, newAlpha)
            } else {
                throw TypeErrorException(s"Let binding has type $t whereas it is bound to expression of type $t1")
            }
        }

        case FunDef(x, t1, e) => {
            val newAlpha = alpha + (x -> t1)
            val t2 = typeOf(e, newAlpha)
            FunType(t1, t2)
        }

        case FunCall(e1, e2) => {
            val ftype = typeOf(e1, alpha)
            ftype match {
                case FunType(t1, t2) => {
                    val argType = typeOf(e2, alpha)
                    if (typeEquals(argType, t1)){
                        t2
                    } else {
                        throw TypeErrorException(s"Call to function with incompatible argument type. Expected $t1, obtained $argType")
                    }
                }
                case _ => { throw TypeErrorException(s"Call to function but with a non function type $ftype")}

            }
        }

        case NewRef(e) => {
            // YOUR CODE HERE
            val argType = typeOf(e, alpha)
            RefType(argType)
        }
        
        case AssignRef(e1, e2) => {
            // YOUR CODE HERE
            val argType = typeOf(e1, alpha)
            val argT = typeOf(e2, alpha)
            argType match {
                case RefType(t) => {
                    if (typeEquals(t, argT)) {UnitType}
                    else {throw TypeErrorException(s"Error")}
                }
                case _ => throw TypeErrorException(s"Error")
                }
        }
        
        case DeRef(e) => {
            // YOUR CODE HERE
            val argType = typeOf(e, alpha)
            argType match {
                case RefType(t) => t
                case _ =>  throw TypeErrorException(s"Error")
            }
        }
        
        case Record(ies) => {
            // YOUR CODE HERE
            val typed = ies.map {
                case(id, expr) => (id, typeOf(expr,alpha))
            }
            RecordType(typed)
        }
    }
}

def typeOfProgram(p: Program) = p match {
    case TopLevel(e) => {
            val t = typeOf(e, Map())
            println(s"Program type computed successfully as $t")
            t
    }
}

-- [E029] Pattern Match Exhaustivity Warning: cmd6.sc:16:4 ---------------------
16 |    e match {
   |    ^
   |    match may not be exhaustive.
   |
   |    It would fail on pattern case: UnitConst
   |
   | longer explanation available when compiling with `-explain`


defined function typeOf
defined function typeOfProgram

In [7]:
//BEGIN TEST

/* 
let x : ref(num) = NewRef(10 ) in 
   let dummy: unit = AssignRef(x, 30) in 
       DeRef(x)
       */

val p1 = Let("x", RefType(NumType), NewRef(Const(10)), Let("dummy", UnitType, AssignRef(Ident("x"), Const(30) ), DeRef(Ident("x"))) )
val t1 = typeOfProgram(TopLevel(p1))
assert(t1 == NumType, "Test 1 failed: answer should be NumType")
passed(6)
//END TEST

Program type computed successfully as NumType

*** Tests Passed (6 points) ***


p1: Let = Let(
  x = "x",
  xType = RefType(t = NumType),
  e1 = NewRef(e = Const(f = 10.0)),
  e2 = Let(
    x = "dummy",
    xType = UnitType,
    e1 = AssignRef(e1 = Ident(s = "x"), e2 = Const(f = 30.0)),
    e2 = DeRef(e = Ident(s = "x"))
  )
)
t1: Type = NumType

In [ ]:
//BEGIN TEST
/* 
let x : ref(num) = NewRef(function(z: num) z + 10) in 
   let dummy: unit = AssignRef(x, 30) in 
       DeRef(x)
       */
val fdef = FunDef("z", NumType, Plus(Ident("z"), Const(10)))
val p2 = Let("x", RefType(NumType), NewRef(fdef), Let("dummy", UnitType, AssignRef(Ident("x"), Const(30) ), DeRef(Ident("x"))) )
val t2 = try {
   typeOfProgram(TopLevel(p2))
   assert(false, "The program should not receive a type")
} catch {
    case TypeErrorException(msg) => s"OK -- caught a type error exception: $msg"
    case e => print(e); assert(false, "Please throw TypeErrorException(message) when a type failure occurs")
}
passed(6)
//END TEST

In [8]:
//BEGIN TEST
/* 
let x : ref(num => num) = NewRef(function(z: num) z + 10) in 
   let dummy: unit = AssignRef(NewRef(35), 30) in 
       DeRef(x)
       */
val fdef = FunDef("z", NumType, Plus(Ident("z"), Const(10)))
val p4 = Let("x", RefType(FunType(NumType, NumType)), NewRef(fdef), Let("dummy", UnitType, AssignRef(NewRef(Const(35)), Const(30) ), DeRef(Ident("x"))) )
val t4 =  typeOfProgram(TopLevel(p4))
assert(t4 == FunType(NumType, NumType), "Test failed")
passed(6)
//END TEST

Program type computed successfully as FunType(NumType,NumType)

*** Tests Passed (6 points) ***


fdef: FunDef = FunDef(
  id = "z",
  idType = NumType,
  e = Plus(e1 = Ident(s = "z"), e2 = Const(f = 10.0))
)
p4: Let = Let(
  x = "x",
  xType = RefType(t = FunType(t1 = NumType, t2 = NumType)),
  e1 = NewRef(
    e = FunDef(
      id = "z",
      idType = NumType,
      e = Plus(e1 = Ident(s = "z"), e2 = Const(f = 10.0))
    )
  ),
  e2 = Let(
    x = "dummy",
    xType = UnitType,
    e1 = AssignRef(e1 = NewRef(e = Const(f = 35.0)), e2 = Const(f = 30.0)),
    e2 = DeRef(e = Ident(s = "x"))
  )
)
t4: Type = FunType(t1 = NumType, t2 = NumType)

In [9]:
//BEGIN TEST
/* 
let x : ref(num => num) = NewRef(function(z: num) z + 10) in 
   let dummy: num = AssignRef(x, 30) in 
       DeRef(x)
       */
val fdef = FunDef("z", NumType, Plus(Ident("z"), Const(10)))
val p3 = Let("x", RefType(FunType(NumType, NumType)), NewRef(fdef), Let("dummy", NumType, AssignRef(Ident("x"), Const(30) ), DeRef(Ident("x"))) )
val t3 = try {
   typeOfProgram(TopLevel(p3))
   assert(false, "The program should not receive a type")
} catch {
    case TypeErrorException(msg) => s"OK -- caught a type error exception: $msg"
    case e => print(e); assert(false, "Please throw TypeErrorException(message) when a type failure occurs")
}
passed(6)
//END TEST


*** Tests Passed (6 points) ***


fdef: FunDef = FunDef(
  id = "z",
  idType = NumType,
  e = Plus(e1 = Ident(s = "z"), e2 = Const(f = 10.0))
)
p3: Let = Let(
  x = "x",
  xType = RefType(t = FunType(t1 = NumType, t2 = NumType)),
  e1 = NewRef(
    e = FunDef(
      id = "z",
      idType = NumType,
      e = Plus(e1 = Ident(s = "z"), e2 = Const(f = 10.0))
    )
  ),
  e2 = Let(
    x = "dummy",
    xType = NumType,
    e1 = AssignRef(e1 = Ident(s = "x"), e2 = Const(f = 30.0)),
    e2 = DeRef(e = Ident(s = "x"))
  )
)
t3: String = "OK -- caught a type error exception: Error"

In [10]:
//BEGIN TEST

val p1 = ("foo",Const(10.0))
val t1 = ("foo",NumType)

val p2 = ("bar",Geq(Const(9.0), Const(10.0)))
val t2 = ("bar",BoolType)

val rdec1 = Record(List(p1,p2))
val rtyp1 = RecordType(List(t1,t2))

val prog1 = TopLevel(rdec1)
assert(typeOfProgram(prog1) == rtyp1, "Test failed")
passed(2)

val p3 = ("baz",NewRef(Const(11.0)))
val t3 = ("baz",RefType(NumType))
val p4 = ("qux",rdec1)
val t4 = ("qux", rtyp1)

val rdec2 = Record(List(p3,p4))
val rtyp2 = RecordType(List(t3,t4))
// prog2 = {baz=11.0, qux={foo=10.0, bar=9.0>10.0}}
val prog2 = TopLevel(rdec2)
assert(typeOfProgram(prog2) == rtyp2, "Test failed")
passed(6)


//END TEST

Program type computed successfully as RecordType(List((foo,NumType), (bar,BoolType)))

*** Tests Passed (2 points) ***
Program type computed successfully as RecordType(List((baz,RefType(NumType)), (qux,RecordType(List((foo,NumType), (bar,BoolType))))))

*** Tests Passed (6 points) ***


p1: (String, Const) = ("foo", Const(f = 10.0))
t1: (String, scala.Tuple2[java.lang.String, cmd10.this.cmd5.NumType]) = (
  "foo",
  NumType
)
p2: (String, Geq) = ("bar", Geq(e1 = Const(f = 9.0), e2 = Const(f = 10.0)))
t2: (String, scala.Tuple2[java.lang.String, cmd10.this.cmd5.BoolType]) = (
  "bar",
  BoolType
)
rdec1: Record = Record(
  ies = List(
    ("foo", Const(f = 10.0)),
    ("bar", Geq(e1 = Const(f = 9.0), e2 = Const(f = 10.0)))
  )
)
rtyp1: RecordType = RecordType(ies = List(("foo", NumType), ("bar", BoolType)))
prog1: TopLevel = TopLevel(
  e = Record(
    ies = List(
      ("foo", Const(f = 10.0)),
      ("bar", Geq(e1 = Const(f = 9.0), e2 = Const(f = 10.0)))
    )
  )
)
p3: (String, NewRef) = ("baz", NewRef(e = Const(f = 11.0)))
t3: (String, RefType) = ("baz", RefType(t = NumType))
p4: (String, Record) = (
  "qux",
  Record(
    ies = List(
      ("foo", Const(f = 10.0)),
      ("bar", Geq(e1 = Const(f = 9.0), e2 = Const(f = 10.0)))
    )
  )
)
t4: (String, RecordType) = 

In [11]:
//BEGIN TEST
/*
let x: ref(Num) = NewRef(0) in 
 let f : num => unit = function (z:num) { x := z} in 
   let obj: {set_counter:ref(num => unit)} = {set_counter = NewRef(f)} in
     obj
*/
val fdef = FunDef("z", NumType, AssignRef(Ident("x"), Ident("z")))
val gdef = FunDef("z", UnitType, DeRef(Ident("x")))
val recrd = Record(List(("set_counter",NewRef(Ident("f")))))
val rectyp = RecordType(List(("set_counter",RefType(FunType(NumType,UnitType)))))
val p3 = Let("x", RefType(NumType), NewRef(Const(0.0)), 
             Let("f", FunType(NumType, UnitType), fdef,
                   Let("obj",rectyp, recrd,
                      Ident("obj"))) )
val t = typeOfProgram(TopLevel(p3))
assert(t == rectyp, s"Test failed: answer should be $rectyp")
passed(6)

//END TEST

Program type computed successfully as RecordType(List((set_counter,RefType(FunType(NumType,UnitType)))))

*** Tests Passed (6 points) ***


fdef: FunDef = FunDef(
  id = "z",
  idType = NumType,
  e = AssignRef(e1 = Ident(s = "x"), e2 = Ident(s = "z"))
)
gdef: FunDef = FunDef(
  id = "z",
  idType = UnitType,
  e = DeRef(e = Ident(s = "x"))
)
recrd: Record = Record(ies = List(("set_counter", NewRef(e = Ident(s = "f")))))
rectyp: RecordType = RecordType(
  ies = List(("set_counter", RefType(t = FunType(t1 = NumType, t2 = UnitType))))
)
p3: Let = Let(
  x = "x",
  xType = RefType(t = NumType),
  e1 = NewRef(e = Const(f = 0.0)),
  e2 = Let(
    x = "f",
    xType = FunType(t1 = NumType, t2 = UnitType),
    e1 = FunDef(
      id = "z",
      idType = NumType,
      e = AssignRef(e1 = Ident(s = "x"), e2 = Ident(s = "z"))
    ),
    e2 = Let(
      x = "obj",
      xType = RecordType(
        ies = List(
          ("set_counter", RefType(t = FunType(t1 = NumType, t2 = UnitType)))
        )
      ),
      e1 = Record(ies = List(("set_counter", NewRef(e = Ident(s = "f"))))),
      e2 = Ident(s = "obj")
    )
  )
)
t: Type = Record

In [12]:
//BEGIN TEST
/*
let x: ref(Num) = NewRef(0) in 
 let f : num => unit = function (z:num) { x := z} in 
  let g : unit => num = function (z:unit) {DeRef(x)} in
   let obj: {set_counter:num => unit, get_counter:unit=>num} = {set_counter = f, get_counter=g} in
     obj
*/
val fdef = FunDef("z", NumType, AssignRef(Ident("x"), Ident("z")))
val gdef = FunDef("z", UnitType, DeRef(Ident("x")))
val recrd = Record(List(("set_counter",Ident("f")), ("get_counter",Ident("g"))))
val rectyp = RecordType(List(("set_counter",FunType(NumType,UnitType)), 
                             ("get_counter",FunType(UnitType,NumType))))
val p3 = Let("x", RefType(NumType), NewRef(Const(0.0)), 
             Let("f", FunType(NumType, UnitType), fdef,
                Let("g", FunType(UnitType, NumType), gdef,
                   Let("obj",rectyp, recrd,
                      Ident("obj")))) )
val t = typeOfProgram(TopLevel(p3))
assert(t == rectyp, s"Test failed: answer should be $rectyp")
passed(6)

//END TEST

Program type computed successfully as RecordType(List((set_counter,FunType(NumType,UnitType)), (get_counter,FunType(UnitType,NumType))))

*** Tests Passed (6 points) ***


fdef: FunDef = FunDef(
  id = "z",
  idType = NumType,
  e = AssignRef(e1 = Ident(s = "x"), e2 = Ident(s = "z"))
)
gdef: FunDef = FunDef(
  id = "z",
  idType = UnitType,
  e = DeRef(e = Ident(s = "x"))
)
recrd: Record = Record(
  ies = List(("set_counter", Ident(s = "f")), ("get_counter", Ident(s = "g")))
)
rectyp: RecordType = RecordType(
  ies = List(
    ("set_counter", FunType(t1 = NumType, t2 = UnitType)),
    ("get_counter", FunType(t1 = UnitType, t2 = NumType))
  )
)
p3: Let = Let(
  x = "x",
  xType = RefType(t = NumType),
  e1 = NewRef(e = Const(f = 0.0)),
  e2 = Let(
    x = "f",
    xType = FunType(t1 = NumType, t2 = UnitType),
    e1 = FunDef(
      id = "z",
      idType = NumType,
      e = AssignRef(e1 = Ident(s = "x"), e2 = Ident(s = "z"))
    ),
    e2 = Let(
      x = "g",
      xType = FunType(t1 = UnitType, t2 = NumType),
      e1 = FunDef(id = "z", idType = UnitType, e = DeRef(e = Ident(s = "x"))),
      e2 = Let(
        x = "obj",
        xType = RecordType(
   

## That's all folks!